# AI Learning Material Analyzer - PDF -> OCR -> Summary -> Quiz demo
This notebook clones the GitHub project and runs the pipeline directly on the service layer.

**Before running:** replace `YOUR_USERNAME` in step 1 with your GitHub username.

## 1. Setup

In [ ]:
!apt-get install -y -q tesseract-ocr
!git clone https://github.com/YOUR_USERNAME/ai-learning-material-analyzer.git
%cd ai-learning-material-analyzer
!pip install -q -r requirements.txt

## 2. Configuration (secrets are typed in, never stored in code)

In [ ]:
import os
from getpass import getpass

os.environ['JWT_SECRET'] = 'colab-demo-secret'  # not used by the pipeline, only required by the config
os.environ['AI_API_KEY'] = getpass('Paste your Gemini API key: ')

## 3. Create two sample PDFs: one with real text, one scanned (image only)

In [ ]:
import fitz

TEXT = (
    "Agentic AI refers to AI systems that can plan, use tools and act toward a goal with little supervision. "
    "An agent observes its environment, decides on an action, executes it, and learns from the result. "
    "Key parts of an agent are memory, planning, tool use and reflection."
)

# Text PDF
doc = fitz.open()
doc.new_page().insert_textbox(fitz.Rect(50, 50, 550, 400), TEXT, fontsize=12)
doc.save('text.pdf')

# Scanned PDF: render the page to an image, then build a PDF that contains only the image
pix = doc[0].get_pixmap(dpi=150)
scanned = fitz.open()
page = scanned.new_page(width=pix.width, height=pix.height)
page.insert_image(page.rect, pixmap=pix)
scanned.save('scanned.pdf')
print('PDFs created')

## 4. Text extraction: OCR must NOT run for the text PDF, and MUST run for the scanned one

In [ ]:
from app.services import pdf_service

for name in ['text.pdf', 'scanned.pdf']:
    result = pdf_service.extract_text(name, max_ocr_pages=20)
    print(f'{name}: used_ocr={result.used_ocr}')
    print(result.text[:200], '\n')

## 5. AI summary

In [ ]:
from app.services import ai_service

text = pdf_service.extract_text('scanned.pdf', max_ocr_pages=20).text
print(ai_service.summarize(text))

## 6. AI quiz (validated JSON)

In [ ]:
from app.services import quiz_service

quiz = quiz_service.generate_quiz(text)
print(quiz.model_dump_json(indent=2))

## 7. Run the automated tests

In [ ]:
!pytest -q